# WavLM Extraction from Google Drive


In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/gdrive')
print('Drive mounted!')

In [ ]:
# Install
!pip install -q transformers accelerate librosa
import os, json, numpy as np
from tqdm import tqdm
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')

In [ ]:
# Find audio files
BASE_DIR = '/content/gdrive/MyDrive/chuckle_audio_all'
folders = ['audio', 'audio_all', 'audio_final', 'audio_new']
audio_files = []
for folder in folders:
    folder_path = os.path.join(BASE_DIR, folder)
    if os.path.exists(folder_path):
        for f in os.listdir(folder_path):
            if f.endswith(('.wav', '.mp3')):
                vid = f.rsplit('.', 1)[0]
                audio_files.append((os.path.join(folder_path, f), vid))
print(f'Total audio files: {len(audio_files)}')

In [ ]:
# Load model
from transformers import WavLMModel
model = WavLMModel.from_pretrained('microsoft/wavlm-base')
model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f'Model on: {device}')

In [ ]:
# Setup output
OUTPUT_DIR = '/content/gdrive/MyDrive/wavlm_embeddings'
os.makedirs(OUTPUT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, 'checkpoint.json')
if os.path.exists(CHECKPOINT_FILE):
    with open(CHECKPOINT_FILE, 'r') as f:
        checkpoint = json.load(f)
    done_ids = set(checkpoint.get('extracted_ids', []))
else:
    done_ids = set()
print(f'Already done: {len(done_ids)}')

In [ ]:
# Extract
import librosa
def load_audio(path, sr=16000):
    try:
        audio, _ = librosa.load(path, sr=sr, duration=3600)
        return audio
    except:
        return None
def extract_embedding(audio, model, device):
    if audio is None or len(audio) < 320:
        return None
    embeddings = []
    chunk_size = 16000 * 30
    for i in range(0, len(audio), chunk_size):
        chunk = audio[i:i+chunk_size]
        if len(chunk) < 320:
            continue
        with torch.no_grad():
            input_tensor = torch.FloatTensor(chunk).unsqueeze(0).to(device)
            outputs = model(input_tensor)
            emb = outputs.last_hidden_state.mean(dim=1).squeeze().cpu().numpy()
            embeddings.append(emb)
    if not embeddings:
        return None
    return np.mean(embeddings, axis=0)
results = []
errors = []
for i, (path, vid) in enumerate(tqdm(audio_files)):
    if vid in done_ids:
        continue
    out_path = os.path.join(OUTPUT_DIR, f'{vid}.npy')
    try:
        audio = load_audio(path)
        emb = extract_embedding(audio, model, device)
        if emb is not None:
            np.save(out_path, emb)
            results.append(vid)
            done_ids.add(vid)
    except:
        errors.append(vid)
    if (i+1) % 25 == 0:
        with open(CHECKPOINT_FILE, 'w') as f:
            json.dump({'extracted': len(results), 'extracted_ids': list(done_ids), 'errors': len(errors)}, f)
        print(f'Checkpoint: {len(results)} extracted')
with open(CHECKPOINT_FILE, 'w') as f:
    json.dump({'extracted': len(results), 'extracted_ids': list(done_ids), 'errors': len(errors), 'completed': True}, f)
print(f'Done! Extracted: {len(results)}, Errors: {len(errors)}')

In [ ]:
# Save checkpoint
import shutil
shutil.make_archive('/content/gdrive/MyDrive/wavlm_embeddings', 'zip', OUTPUT_DIR)
print('Saved: /content/gdrive/MyDrive/wavlm_embeddings.zip')